### Импорты и подготовка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# воспроизводимость
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем: {device}")

#### Нормализация

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)

#### Нарезка на скользящие окна

In [ ]:
LSTM работает с последовательностями, поэтому режем ряд на окна.

def create_sequences(data, seq_len, step=1):
    """Превращает [N, features] в [num_windows, seq_len, features]"""
    sequences = []
    indices = []  # запоминаем конец окна (для привязки ко времени)
    for i in range(0, len(data) - seq_len + 1, step):
        sequences.append(data[i:i + seq_len])
        indices.append(i + seq_len - 1)
    return np.array(sequences), np.array(indices)

SEQ_LEN = 30   # длина окна (подберите под частоту ваших данных)
X_seq, seq_idx = create_sequences(X_scaled, SEQ_LEN, step=1)
print("Форма окон:", X_seq.shape)  # [num_windows, SEQ_LEN, n_features]

#### Модель LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=32, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features

        # Encoder: сжимает последовательность в вектор
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )
        # Decoder: восстанавливает последовательность
        self.decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )
        self.output_layer = nn.Linear(embedding_dim, n_features)

    def forward(self, x):
        # Кодирование
        _, (hidden, _) = self.encoder(x)         # hidden: [1, batch, emb_dim]
        # Повторяем скрытый вектор на всю длину последовательности
        latent = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        # Декодирование
        decoded, _ = self.decoder(latent)
        out = self.output_layer(decoded)
        return out

In [ ]:
# DataLoader
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = LSTMAutoencoder(
    n_features=len(features),
    embedding_dim=32,
    seq_len=SEQ_LEN
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss: {avg_loss:.5f}")

# График обучения
plt.figure(figsize=(8,3))
plt.plot(history)
plt.title('Loss при обучении')
plt.xlabel('Эпоха'); plt.ylabel('MSE')
plt.show()

#### Подсчёт ошибки реконструкции и поиск аномалий

In [ ]:
model.eval()
errors = []

with torch.no_grad():
    for i in range(0, len(X_tensor), 256):
        batch = X_tensor[i:i+256].to(device)
        recon = model(batch)
        # ошибка для каждого окна (усредняем по времени и признакам)
        err = torch.mean((recon - batch)**2, dim=(1, 2))
        errors.extend(err.cpu().numpy())

errors = np.array(errors)

# Порог: например, 99-й перцентиль ошибки
threshold = np.percentile(errors, 99)
print(f"Порог: {threshold:.4f}")

anomaly_windows = errors > threshold

# Привязываем аномалии обратно к строкам датафрейма
df['recon_error'] = np.nan
df.loc[seq_idx, 'recon_error'] = errors
df['anomaly'] = False
df.loc[seq_idx[anomaly_windows], 'anomaly'] = True

print(f"Найдено аномальных точек: {df['anomaly'].sum()}")

#### Визуализация — ошибка реконструкции

In [ ]:
plt.figure(figsize=(15, 4))
plt.plot(seq_idx, errors, label='Ошибка реконструкции', color='steelblue')
plt.axhline(threshold, color='red', linestyle='--', label='Порог')
plt.scatter(seq_idx[anomaly_windows], errors[anomaly_windows],
            color='red', s=20, label='Аномалии')
plt.title('Ошибка реконструкции по времени')
plt.xlabel('Индекс времени'); plt.ylabel('MSE')
plt.legend()
plt.show()

#### Визуализация — аномалии на признаках

In [ ]:
# Покажем 4 главных признака с подсветкой аномалий
plot_features = ['высота', 'скорость', 'тангаж', 'крен']

fig, axes = plt.subplots(len(plot_features), 1, figsize=(15, 10), sharex=True)

for ax, col in zip(axes, plot_features):
    ax.plot(df.index, df[col], color='steelblue', lw=0.8)
    anom = df[df['anomaly']]
    ax.scatter(anom.index, anom[col], color='red', s=15, zorder=5)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)

axes[0].set_title('Аномалии на признаках (красные точки)')
axes[-1].set_xlabel('Время')
plt.tight_layout()
plt.show()

#### Визуализация — PCA (общая картина)

In [ ]:
# Берём по одной точке на окно (последняя точка окна)
X_points = X_scaled[seq_idx]
pca = PCA(n_components=2)
emb = pca.fit_transform(X_points)

plt.figure(figsize=(8, 6))
plt.scatter(emb[~anomaly_windows, 0], emb[~anomaly_windows, 1],
            c='steelblue', s=10, label='Норма', alpha=0.5)
plt.scatter(emb[anomaly_windows, 0], emb[anomaly_windows, 1],
            c='red', s=30, label='Аномалия')
plt.title('PCA проекция (2D)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend()
plt.show()

#### Heatmap — какие признаки "виноваты"

In [ ]:
Покажет, по каким именно признакам была наибольшая ошибка.

# Считаем ошибку по каждому признаку отдельно для аномальных окон
model.eval()
with torch.no_grad():
    anom_idx = np.where(anomaly_windows)[0]
    if len(anom_idx) > 0:
        sample = X_tensor[anom_idx].to(device)
        recon = model(sample)
        # ошибка по признакам: усредняем по времени
        feat_error = torch.mean((recon - sample)**2, dim=1).cpu().numpy()

        plt.figure(figsize=(12, 6))
        sns.heatmap(feat_error.T, cmap='Reds',
                    yticklabels=features,
                    cbar_kws={'label': 'Ошибка'})
        plt.title('Вклад каждого признака в аномалии')
        plt.xlabel('Номер аномального окна')
        plt.ylabel('Признак')
        plt.tight_layout()
        plt.show()

In [ ]:
Важные замечания


Параметр	Что делать
SEQ_LEN	Подберите под частоту данных. Если запись 1 Гц — окно 30 = 30 сек
threshold (перцентиль)	99% — старт. Меньше процент → больше аномалий
embedding_dim	Слишком большой → модель запомнит всё, включая аномалии
Разные полёты	Не смешивайте! Режьте окна внутри одного полёта


In [ ]:
# производные — как быстро меняются признаки
for col in ['высота', 'скорость', 'тангаж', 'крен']:
    df[f'{col}_diff'] = df.groupby('aircraft_id')[col].diff()

# заполняем NaN (первая строка каждого аппарата)
df = df.fillna(0)

# добавляем новые признаки в список
features = features + ['высота_diff', 'скорость_diff', 'тангаж_diff', 'крен_diff']

In [ ]:
# 5. Нарезка окон ПО КАЖДОМУ АППАРАТУ
def create_sequences_grouped(df, features, seq_len, id_column, step=1):
    all_seq, all_idx, all_ids = [], [], []
    for aid, group in df.groupby(id_column):
        group = group.sort_index()
        data = group[features].values
        gidx = group.index.values
        for i in range(0, len(data) - seq_len + 1, step):
            all_seq.append(data[i:i + seq_len])
            all_idx.append(gidx[i + seq_len - 1])
            all_ids.append(aid)
    return np.array(all_seq), np.array(all_idx), np.array(all_ids)

SEQ_LEN = 30
X_seq, seq_idx, seq_ids = create_sequences_grouped(
    df, features, SEQ_LEN, id_column='aircraft_id'
)
print("Окон:", X_seq.shape)